# Freqtrade Strategy001 (upstream) — ارزیابی استراتژی 5m پین‌شده

**فقط در Google Colab اجرا شود.** نصب وابستگی‌ها فقط داخل runtime کولب انجام می‌شود؛ روی ماشین محلی MT5 اجرا/نصب نکنید.

- این Strategy001 استراتژی **5m** واقعی upstream است؛ آن را به‌عنوان momentum تیک M1 برچسب نزنید.
- سورس GPL-3.0 در این ریپو vendored/بازمجوز نشده؛ در Colab از ریپوی upstream در commit پین‌شده clone می‌شود و SHA256 قبل از اجرا بررسی می‌شود.
- خروجی جعلی تولید نکنید. تا اجرای واقعی انجام نشود، «اجراشده» محسوب نمی‌شود.
- هیچ سرور عمومی، هیچ اتصال MT5، هیچ سفارش/guardian.
- داده ورودی OHLCV redact‌شده 5m (بدون credential/account/journal).
- پین: `freqtrade/freqtrade-strategies` commit `f3340ce11f5bdf62f598522e64d1f5638eaa13f5`، Strategy001 SHA256 `2357996e6594458e8db45b5f757918462afa3c998de9d06e2af0c009ddc4b80d`.


In [ ]:
# COLAB ONLY
import os, sys, subprocess, json, hashlib, math
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
assert IN_COLAB, 'This notebook must run only inside Google Colab runtime.'
FREQTRADE_COMMIT='f3340ce11f5bdf62f598522e64d1f5638eaa13f5'
STRATEGY001_SHA256='2357996e6594458e8db45b5f757918462afa3c998de9d06e2af0c009ddc4b80d'
WORK='/content/freqtrade-work'
REPO=f'{WORK}/freqtrade-strategies'
os.makedirs(WORK, exist_ok=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','pandas','numpy','ta-lib','technical'])
# freqtrade package may be needed for IStrategy base; install in Colab runtime only
subprocess.check_call([sys.executable,'-m','pip','install','-q','freqtrade'])
if not os.path.exists(REPO):
    subprocess.check_call(['git','clone','https://github.com/freqtrade/freqtrade-strategies.git',REPO])
subprocess.check_call(['git','-C',REPO,'fetch','--all'])
subprocess.check_call(['git','-C',REPO,'checkout',FREQTRADE_COMMIT])
source_path = os.path.join(REPO,'user_data','strategies','Strategy001.py')
actual = hashlib.sha256(open(source_path,'rb').read()).hexdigest()
assert actual==STRATEGY001_SHA256, f'Unverified Strategy001 source: {actual}'
print('Verified Strategy001 source at pinned commit')


## 2) آپلود داده OHLCV redact‌شده 5m
فرمت: `{"symbol":"XAUUSD","rows":[[epoch_sec,open,high,low,close,volume], ...]}` با کندل‌های 5m بسته و پیوسته (>=200). این استراتژی 5m است؛ M1 tick momentum نیست.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert uploaded, 'No file uploaded'
req = json.loads(uploaded[next(iter(uploaded))].decode('utf-8'))
symbol=str(req.get('symbol','XAUUSD'))
rows=req['rows']
forbidden={'login','password','token','secret','account','journal','credential'}
assert not any(k in json.dumps(req).lower() for k in forbidden), 'Redact sensitive fields first'
assert len(rows)>=200, 'Need >=200 contiguous closed 5m bars'
times=[int(r[0]) for r in rows]
assert all(b-a==300 for a,b in zip(times,times[1:])), 'Rows must be contiguous 5m bars'
print('Loaded', len(rows), '5m rows for', symbol)


## 3) اجرای واقعی upstream Strategy001
قبل از اجرا hash تأیید شده است. فقط متدهای واقعی `populate_indicators/populate_entry_trend/populate_exit_trend` استفاده می‌شوند.

In [ ]:
import importlib.util, pandas as pd
spec = importlib.util.spec_from_file_location('verified_upstream_strategy001', source_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
strategy = module.Strategy001({})
df = pd.DataFrame(rows, columns=['time','open','high','low','close','volume'])
df['date'] = pd.to_datetime(df.pop('time'), unit='s', utc=True)
metadata = {'pair': f'{symbol}/RESEARCH'}
df = strategy.populate_indicators(df.copy(), metadata)
df = strategy.populate_entry_trend(df.copy(), metadata)
df = strategy.populate_exit_trend(df.copy(), metadata)
last = df.iloc[-1]
signal = {
  'source':'freqtrade/freqtrade-strategies/Strategy001',
  'upstream_commit':FREQTRADE_COMMIT,
  'license':'GPL-3.0',
  'strategy_timeframe':'5m',
  'is_m1_tick_momentum': False,
  'enter_long': bool(last.get('enter_long',0)==1),
  'exit_long': bool(last.get('exit_long',0)==1),
  'execution_authorized': False,
}
print(json.dumps(signal, indent=2, ensure_ascii=False))


## 4) ارزیابی hold-out و مقایسه baseline (تحقیقاتی، نه اثبات سودآوری)
بخش پایانی به‌عنوان hold-out در نظر گرفته می‌شود؛ سیگنال ورود Strategy001 با baseline بدون سیگنال مقایسه می‌شود. این ارزیابی تحقیقاتی است، نه مجوز معامله MT5.

In [ ]:
# simple research hold-out: last 20% bars, 5m forward close return after entry_long signals
n=len(df); split=max(200, int(n*0.8))
test = df.iloc[split:].copy().reset_index(drop=True)
fwd=3  # 15m forward on 5m bars, research-only
entries=test.index[test.get('enter_long',0)==1].tolist()
entry_rets=[]
for i in entries:
    j=i+fwd
    if j < len(test):
        entry_rets.append(float(test.loc[j,'close'])/float(test.loc[i,'close'])-1)
baseline_rets=[]
for i in range(0, len(test)-fwd):
    baseline_rets.append(float(test.loc[i+fwd,'close'])/float(test.loc[i,'close'])-1)
def mean(x): return sum(x)/len(x) if x else None
report = {
  **signal,
  'evaluation': {
    'holdout_bars': int(len(test)),
    'forward_bars': fwd,
    'entry_signal_count': len(entries),
    'mean_forward_return_after_entries': mean(entry_rets),
    'mean_forward_return_baseline': mean(baseline_rets),
    'validated_for_trading': False,
    'warning': 'Research-only holdout; not profitability proof; not MT5 execution authorization; Strategy001 is 5m, not M1 tick momentum.'
  }
}
print(json.dumps(report, indent=2, ensure_ascii=False))


## 5) دانلود artifact تحقیقاتی (سازگار با adapter trade-mcp)
خروجی JSON شامل سیگنال/متادیتای upstream و ارزیابی تحقیقاتی است؛ `execution_authorized=false`.

In [ ]:
out_path=f'/content/freqtrade_strategy001_{symbol}_5m_research.json'
with open(out_path,'w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
from google.colab import files
files.download(out_path)
print('Downloaded. Keep local/owner-only; do not expose as public server; do not run MT5 orders.')


## محدودیت‌ها
- Strategy001 5m است؛ با momentum تیک M1 یکسان نیست و مجوز اجرای MT5 ندارد.
- مجوز GPL-3.0 در توزیع/ترکیب اعمال می‌شود؛ سورس upstream vendored نشده.
- عدم اجرای نوت‌بوک = عدم تکمیل اجرای واقعی ابری.